In [1]:
import numpy as np
from numpy.linalg import inv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2
sns.set()

from sklearn.datasets import make_spd_matrix

%matplotlib inline

# fix random seed
np.random.seed(10)

In [2]:
def is_pos_def(A):
    if is_symmetric(A):
        try:
            np.linalg.cholesky(A)
            return True
        except np.linalg.LinAlgError:
            return False
    else:
        return False

def is_symmetric(a, rtol=1e-05, atol=1e-08):
    return np.allclose(a, a.T, rtol=rtol, atol=atol)

In [33]:
def calc_matrices_precision(psi_hat, H_s, C, dim):
    A = np.zeros((dim, dim))
    B = np.zeros((dim,))
    inv_psi_hat = inv(psi_hat) # calculate psi_hat inverse
    for g in range(dim):
        for h in range(dim):
            mult = ((inv_psi_hat.dot(H_s[g])).dot(inv_psi_hat)).dot(H_s[h])
            lhs = np.trace(mult)
            A[g, h] = lhs
        rhs = np.trace(inv_psi_hat.dot(H_s[g])) - np.trace(C.dot(H_s[g]))
        B[g] = rhs
        
    return A, B

def iterative_soln_precision(coeffs_zero, H_s, C, dim, iters=5):
    s_imo = coeffs_zero
    for it in range(iters):
        psi_hat = (s_imo.reshape(-1, 1, 1)*H_s).sum(0) # calculate psi_hat
        A, B = calc_matrices_precision(psi_hat, H_s, C, dim)
        t_i = inv(A).dot(B)
        s_i = s_imo + t_i
        print(s_i)
        s_imo = s_i
        
    return s_imo

def iterative_soln_precision_single(coeffs_zero, H_s, C, dim, modify_index=0, iters=5):
    s_imo = coeffs_zero
    for it in range(iters):
        psi_hat = (s_imo.reshape(-1, 1, 1)*H_s).sum(0) # calculate psi_hat
        A, B = calc_matrices_precision(psi_hat, H_s, C, dim)
        t_i = inv(A).dot(B)
        s_i = s_imo.copy()
        s_i[modify_index] = s_imo[modify_index] + t_i[modify_index]
        #print(s_i)
        s_imo = s_i
        
    return s_imo

def unbiased_init_precision(C, H_s, N, dim):
    A = np.zeros((dim, dim))
    B = np.zeros((dim, ))
    for g in range(dim):
        for h in range(dim):
            A[g, h] = np.trace(H_s[g].dot(H_s[h]))
        B[g] = np.trace(inv(C).dot(H_s[g]))
    return inv(A).dot(B)

def calc_likelihood_precision(coeffs, H_s, C, N, P):
    precision_hat = (coeffs.reshape(-1, 1, 1)*H_s).sum(0)
    log_l = -P*np.log(2*np.pi) + np.log(np.linalg.det(precision_hat)) - np.trace(np.dot(precision_hat, C))
    
    return log_l*(N/2)

def generate_matrices(prec_coeffs=None, M=2, dim=4):
    H_s = []
    if not prec_coeffs:
        prec_coeffs = np.random.rand(M)
    precision = np.zeros((dim, dim))
    for i in range(M):
        mat = make_spd_matrix(dim)
        H_s.append(mat)
        precision += prec_coeffs[i]*mat
    H_s = np.array(H_s)
    
    return H_s, precision, prec_coeffs

def collect_precision_matrix(H_s, prec_coeffs):
    precision = (prec_coeffs.reshape(-1, 1, 1)*H_s).sum(0)
    
    return precision

def sim_data(covar, dim, N=1000):
    assert is_symmetric(covar), is_pos_def(covar)
    data_sim = np.random.multivariate_normal(np.zeros(dim), covar, N).T
    C = np.cov(data_sim)
    
    return data_sim, C

def likelihood_ratio_test(likelihood_null, likelihood_alternative, dof):
    delta_d = -2*(likelihood_null-likelihood_alternative)
    
    return delta_d, chi2.pdf(delta_d, dof)

In [34]:
M = 2
dim = 4
N = 1000

H_s, precision_one, prec_coeffs_one = generate_matrices(M=M, dim=dim)
data_one, C_one = sim_data(covar=inv(precision_one), dim=dim, N=N)

In [35]:
s_zero_one = unbiased_init_precision(C_one, H_s, N=N, dim=M)
coeffs_hat_one = iterative_soln_precision(s_zero_one, H_s, C_one, dim=M, iters=25)
coeffs_hat_one, prec_coeffs_one

[0.78600931 0.71599592]
[0.78561914 0.71929955]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]
[0.78561499 0.71931207]


(array([0.78561499, 0.71931207]), array([0.76908704, 0.68844043]))

In [36]:
prec_coeffs_two = prec_coeffs_one.copy()
prec_coeffs_two[0] += 0.1
precision_two = collect_precision_matrix(H_s, prec_coeffs_two)
data_two, C_two = sim_data(covar=inv(precision_two), dim=dim, N=N)

In [37]:
s_zero_two = unbiased_init_precision(C_two, H_s, N=N, dim=M)
coeffs_hat_two = iterative_soln_precision(s_zero_two, H_s, C_two, dim=M, iters=25)
coeffs_hat_two, prec_coeffs_two

[0.86536628 0.69636218]
[0.86519073 0.69763594]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]
[0.86519005 0.69763782]


(array([0.86519005, 0.69763782]), array([0.86908704, 0.68844043]))

In [38]:
data_total = np.concatenate((data_one, data_two), axis=1)
C_total = np.cov(data_total)
s_zero_total = unbiased_init_precision(C_total, H_s, N=N, dim=M)
coeffs_hat_total = iterative_soln_precision(s_zero_total, H_s, C_total, dim=M, iters=25)
coeffs_hat_total

[0.82390131 0.7085653 ]
[0.82387901 0.70880723]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]
[0.82387899 0.7088073 ]


array([0.82387899, 0.7088073 ])

In [39]:
"""
LRT Using total data to fit null model
"""

# covariance from total data/model - sample size 2*N
null_likelihood = calc_likelihood_precision(coeffs=coeffs_hat_total, H_s=H_s, C=C_total, 
                                            N=2*N, P=dim)
# covariance from first data, total model
alt_likelihood_one = calc_likelihood_precision(coeffs=coeffs_hat_total, H_s=H_s, C=C_one,
                                               N=N, P=dim)
# covariance from second model, second data
alt_likelihood_two = calc_likelihood_precision(coeffs=coeffs_hat_two, H_s=H_s, C=C_two, 
                                              N=N, P=dim)

likelihood_ratio_test(null_likelihood, alt_likelihood_one+alt_likelihood_two, M)

(2.115521133848233, 0.173616272164166)

In [40]:
"""
LRT Using only first data to fit null model
"""

# covariance from first model, first data 
null_likelihood = calc_likelihood_precision(coeffs=coeffs_hat_one, H_s=H_s, C=C_one,
                                               N=N, P=dim)
# covariance from second model, second data
alt_likelihood = calc_likelihood_precision(coeffs=coeffs_hat_two, H_s=H_s, C=C_two, 
                                              N=N, P=dim)

likelihood_ratio_test(null_likelihood, alt_likelihood, M)

(126.39219050492648, 1.791655711490682e-28)

In [42]:
"""
Refit model on second data using TOTAL model's init

Run two LRT - one for each coefficient change
"""

# modify the first index of coeffs, use second data
mod_first_coeffs = iterative_soln_precision_single(coeffs_hat_total, H_s, 
                                                   C=C_two, dim=M, modify_index=0, iters=15)
# modify the second index of coeffs, use second data
mod_second_coeffs = iterative_soln_precision_single(coeffs_hat_total, H_s, 
                                                    C=C_two, dim=M, modify_index=1, iters=15)

# covariance from total data/model - sample size 2*N
null_likelihood = calc_likelihood_precision(coeffs=coeffs_hat_total, H_s=H_s, C=C_total, 
                                            N=2*N, P=dim)
# covariance from first data, total model
alt_likelihood_one = calc_likelihood_precision(coeffs=coeffs_hat_total, H_s=H_s, C=C_one,
                                               N=N, P=dim)
# covariance from second model, second data, modified first coefficient
alt_likelihood_two_first = calc_likelihood_precision(coeffs=mod_first_coeffs, H_s=H_s, 
                                                     C=C_two, N=N, P=dim)
# covariance from second model, second data, modified second coefficient
alt_likelihood_two_second = calc_likelihood_precision(coeffs=mod_second_coeffs, H_s=H_s, 
                                                      C=C_two, N=N, P=dim)
first_test = likelihood_ratio_test(null_likelihood, 
                                   alt_likelihood_one+alt_likelihood_two_first, M)
second_test = likelihood_ratio_test(null_likelihood,
                                   alt_likelihood_one+alt_likelihood_two_second, M)
first_test, second_test

((1.95128758357896, 0.18847479918658094),
 (0.6202689956480754, 0.36667415791911867))

In [43]:
"""
Refit model on second data using FIRST model's init

Run two LRT - one for each coefficient change
"""

# modify the first index of coeffs, use second data
mod_first_coeffs = iterative_soln_precision_single(coeffs_hat_one, H_s, 
                                                   C=C_two, dim=M, modify_index=0, iters=15)
# modify the second index of coeffs, use second data
mod_second_coeffs = iterative_soln_precision_single(coeffs_hat_one, H_s, 
                                                    C=C_two, dim=M, modify_index=1, iters=15)
# covariance from first data, first model
null_likelihood = calc_likelihood_precision(coeffs=coeffs_hat_one, H_s=H_s, C=C_one,
                                               N=N, P=dim)
# covariance from second model, second data, modified first coefficient
alt_likelihood_first = calc_likelihood_precision(coeffs=mod_first_coeffs, H_s=H_s, 
                                                     C=C_two, N=N, P=dim)
# covariance from second model, second data, modified second coefficient
alt_likelihood_second = calc_likelihood_precision(coeffs=mod_second_coeffs, H_s=H_s, 
                                                      C=C_two, N=N, P=dim)
first_test = likelihood_ratio_test(null_likelihood, 
                                   alt_likelihood_first, M)
second_test = likelihood_ratio_test(null_likelihood,
                                   alt_likelihood_second, M)
first_test, second_test

((125.77626225990753, 2.437820234018923e-28),
 (120.76746101983008, 2.982971109282673e-27))

In [49]:
coeffs_hat_one, mod_first_coeffs, mod_second_coeffs, prec_coeffs_one, prec_coeffs_two

(array([0.78561499, 0.71931207]),
 array([0.86535273, 0.71931207]),
 array([0.78561499, 0.69862287]),
 array([0.76908704, 0.68844043]),
 array([0.86908704, 0.68844043]))